In [1]:
!pip install -q torchaudio librosa pandas requests onnx onnxruntime scikit-learn onnxscript tqdm torch

In [2]:
import os
import pandas as pd
import requests
from tqdm.notebook import tqdm

# 1. Load data
BASE_DIR = os.path.dirname(os.path.abspath('Multilabel_Sound_Classifier.ipynb'))
csv_path = os.path.join(BASE_DIR, 'raw_frog_observations.csv')
df = pd.read_csv(csv_path)

# Determine the correct URL column if 'audio_url' is missing
audio_col = 'audio_url' if 'audio_url' in df.columns else 'url'
name_col = 'common_name' if 'common_name' in df.columns else 'name'

# Filter columns safely
columns_to_keep = ['id', audio_col, name_col]
available_cols = [c for c in columns_to_keep if c in df.columns]
df = df[available_cols].dropna()

# Standardize names for downstream tasks
df = df.rename(columns={audio_col: 'audio_url', name_col: 'common_name'})

display(df.head())


,id,audio_url,common_name
0,248551,http://www.inaturalist.org/observations/248551,Cuban Tree Frog
1,311690,http://www.inaturalist.org/observations/311690,Southern Leopard Frog
2,314642,http://www.inaturalist.org/observations/314642,Southern Leopard Frog
3,318727,http://www.inaturalist.org/observations/318727,Green Frog
4,338303,http://www.inaturalist.org/observations/338303,Coastal Plains Leopard Frog


In [3]:
import os
import pandas as pd
import requests
from tqdm.notebook import tqdm
import time
import torchaudio
import logging

# Configure logging to catch errors without crashing the whole process
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

audio_dir = os.path.join(BASE_DIR, 'frog_audio')
os.makedirs(audio_dir, exist_ok=True)

def get_inaturalist_audio_url(observation_id, max_retries=10, initial_delay=10):
    api_url = f"https://api.inaturalist.org/v1/observations/{observation_id}"
    attempt = 0
    while attempt < max_retries:
        try:
            response = requests.get(api_url, timeout=10)
            response.raise_for_status() # Raise an exception for HTTP errors (including 429)
            data = response.json()

            if 'results' in data and data['results']:
                observation_data = data['results'][0]
                if 'observation_sounds' in observation_data:
                    for osound in observation_data['observation_sounds']:
                        if 'sound' in osound and 'file_url' in osound['sound']:
                            return osound['sound']['file_url']
            # If no sound found in results, or results are empty
            return None # Explicitly return None if no sound URL found in valid response

        except requests.exceptions.RequestException as e:
            # Only print retry message if it's not the last attempt
            if attempt < max_retries - 1:
                logging.warning(f"Attempt {attempt + 1}/{max_retries} failed for API call (Observation ID: {observation_id}, Error: {e}). Retrying in {initial_delay * (2 ** attempt)}s...")
            else:
                logging.error(f"Final attempt {attempt + 1}/{max_retries} failed for API call (Observation ID: {observation_id}, Error: {e}).")
        except Exception as e:
            if attempt < max_retries - 1:
                logging.warning(f"Attempt {attempt + 1}/{max_retries} failed for API call (Observation ID: {observation_id}, Unexpected Error: {e}). Retrying in {initial_delay * (2 ** attempt)}s...")
            else:
                logging.error(f"Final attempt {attempt + 1}/{max_retries} failed for API call (Observation ID: {observation_id}, Unexpected Error: {e}).")

        attempt += 1
        if attempt < max_retries:
            time.sleep(initial_delay * (2 ** (attempt - 1))) # Exponential backoff
    return None


def download_audio(row, output_dir, max_retries=3, initial_delay=1):
    species = str(row.get('common_name', 'Unknown')).replace(' ', '_').replace('/', '_')
    obs_id = row.get('id', 'unknown_id')

    # First, try to get the official audio URL from iNaturalist API
    official_audio_url = get_inaturalist_audio_url(obs_id)

    # CRITICAL CHANGE: If no official_audio_url is found, do NOT fall back to the old URL.
    # Instead, skip the download for this observation.
    url_to_download = official_audio_url

    if not url_to_download:
        logging.info(f"Skipping {species}_{obs_id}.mp3: No valid audio URL found via API.")
        return None

    filename = f"{species}_{obs_id}.mp3"
    filepath = os.path.join(output_dir, filename)

    # Check if file exists and is not empty before re-downloading
    if os.path.exists(filepath) and os.path.getsize(filepath) > 0:
        return filepath

    attempt = 0
    while attempt < max_retries:
        try:
            response = requests.get(url_to_download, timeout=10)
            if response.status_code == 200:
                with open(filepath, 'wb') as f:
                    f.write(response.content)
                return filepath
            else:
                if attempt < max_retries - 1:
                    logging.warning(f"Attempt {attempt + 1}/{max_retries} failed for {filename} (HTTP {response.status_code}). URL: {url_to_download}. Retrying in {initial_delay * (2 ** attempt)}s...")
                else:
                    logging.error(f"Final attempt {attempt + 1}/{max_retries} failed for {filename} (HTTP {response.status_code}). URL: {url_to_download}.")
        except requests.exceptions.RequestException as e:
            if attempt < max_retries - 1:
                logging.warning(f"Attempt {attempt + 1}/{max_retries} failed for {filename} (Network Error: {e}). URL: {url_to_download}. Retrying in {initial_delay * (2 ** attempt)}s...")
            else:
                logging.error(f"Final attempt {attempt + 1}/{max_retries} failed for {filename} (Network Error: {e}). URL: {url_to_download}.")
        except Exception as e:
            if attempt < max_retries - 1:
                logging.warning(f"Attempt {attempt + 1}/{max_retries} failed for {filename} (Unexpected Error: {e}). URL: {url_to_download}. Retrying in {initial_delay * (2 ** attempt)}s...")
            else:
                logging.error(f"Final attempt {attempt + 1}/{max_retries} failed for {filename} (Unexpected Error: {e}). URL: {url_to_download}.")

        attempt += 1
        if attempt < max_retries:
            time.sleep(initial_delay * (2 ** (attempt - 1))) # Exponential backoff

    logging.error(f"Failed to download {filename} after {max_retries} attempts from URL: {url_to_download}.")
    return None

# --- Audio File Validation Function ---
def validate_audio_file(filepath):
    if not os.path.exists(filepath) or os.path.getsize(filepath) == 0:
        return False # File doesn't exist or is empty
    try:
        # Try to load the file using torchaudio to check for decoding errors
        torchaudio.load(filepath, normalize=False)
        return True
    except Exception as e:
        logging.warning(f"Validation failed for {filepath}: {e}")
        os.remove(filepath) # Remove corrupted file
        return False


if 'id' in df.columns:
    initial_rows = len(df)
    tqdm.pandas(desc="Downloading Audio")
    df['temp_local_path'] = df.progress_apply(lambda x: download_audio(x, audio_dir), axis=1)

    # Drop rows where download failed
    df['local_path'] = df['temp_local_path']
    df = df.dropna(subset=['local_path'])
    df = df.drop(columns=['temp_local_path'])

    download_success_count = len(df)
    logging.info(f"Finished initial download phase. Successfully downloaded {download_success_count} files.")

    # --- Post-download validation ---
    logging.info("Starting audio file validation...")
    valid_files_mask = df['local_path'].progress_apply(validate_audio_file)
    initial_valid_df_count = len(df)
    df = df[valid_files_mask]
    validated_count = len(df)
    removed_during_validation = initial_valid_df_count - validated_count
    logging.info(f"Validation complete. Removed {removed_during_validation} corrupted/invalid files.")

    final_downloaded_count = len(df)
    total_failed_count = initial_rows - final_downloaded_count

    print(f"Initial entries in DataFrame: {initial_rows}")
    print(f"Successfully downloaded (or already existed) and validated: {final_downloaded_count} audio files.")
    print(f"Failed to download or were invalid: {total_failed_count} audio files.")
    print(f"Remaining entries in DataFrame after dropping failures and invalid files: {len(df)}")
else:
    print("Skipping download step due to missing 'id' column, which is required for iNaturalist API calls.")


2026-06-08 06:11:47,680 - INFO - Skipping Green_Frog_318727.mp3: No valid audio URL found via API.
2026-06-08 06:11:50,660 - INFO - Skipping Green_Treefrog_353701.mp3: No valid audio URL found via API.
2026-06-08 06:12:10,277 - INFO - Skipping Green_Frog_603245.mp3: No valid audio URL found via API.
2026-06-08 06:12:19,890 - INFO - Skipping Green_Frog_657567.mp3: No valid audio URL found via API.
2026-06-08 06:12:21,788 - INFO - Skipping Green_Frog_682120.mp3: No valid audio URL found via API.
2026-06-08 06:12:22,646 - INFO - Skipping American_Bullfrog_715555.mp3: No valid audio URL found via API.
2026-06-08 06:12:23,379 - INFO - Skipping Green_Frog_715561.mp3: No valid audio URL found via API.
2026-06-08 06:12:29,849 - INFO - Skipping Southern_Leopard_Frog_1272180.mp3: No valid audio URL found via API.
2026-06-08 06:12:34,006 - INFO - Skipping Southern_Leopard_Frog_1290004.mp3: No valid audio URL found via API.
2026-06-08 06:12:39,313 - INFO - Skipping Southern_Leopard_Frog_1376278.mp

2026-06-09 00:44:14,347 - WARNING - Validation failed for /media/crow/My Passport/eFrog Trainer/frog_audio/Cuban_Tree_Frog_248551.mp3: TorchCodec is required for load_with_torchcodec. Please install torchcodec to use this function.
2026-06-09 00:44:14,347 - WARNING - Validation failed for /media/crow/My Passport/eFrog Trainer/frog_audio/Southern_Leopard_Frog_311690.mp3: TorchCodec is required for load_with_torchcodec. Please install torchcodec to use this function.
2026-06-09 00:44:14,348 - WARNING - Validation failed for /media/crow/My Passport/eFrog Trainer/frog_audio/Southern_Leopard_Frog_314642.mp3: TorchCodec is required for load_with_torchcodec. Please install torchcodec to use this function.
2026-06-09 00:44:14,348 - WARNING - Validation failed for /media/crow/My Passport/eFrog Trainer/frog_audio/Coastal_Plains_Leopard_Frog_338303.mp3: TorchCodec is required for load_with_torchcodec. Please install torchcodec to use this function.
2026-06-09 00:44:14,349 - WARNING - Validation f

Initial entries in DataFrame: 10098
Successfully downloaded (or already existed) and validated: 0 audio files.
Failed to download or were invalid: 10098 audio files.
Remaining entries in DataFrame after dropping failures and invalid files: 0


### Smoketest: Verify Audio Files Downloaded

In [ ]:
import os

# Get a list of all files in the audio directory
downloaded_files = os.listdir(audio_dir)

print(f"Contents of {audio_dir}:")
print(f"Total files found: {len(downloaded_files)}")

# Optionally, print the first few file names to confirm
if len(downloaded_files) > 0:
    print("First 5 downloaded files:")
    for i, file_name in enumerate(downloaded_files[:5]):
        file_path = os.path.join(audio_dir, file_name)
        file_size_kb = os.path.getsize(file_path) / 1024 # in KB
        print(f"- {file_name} ({file_size_kb:.2f} KB)")
else:
    print("No files found in the directory.")

# A basic assertion to check if files exist
assert len(downloaded_files) > 0, "ERROR: No audio files were found in the download directory!"
print("Smoketest passed: Audio files are present in the directory.")

In [ ]:
import numpy as np
import pandas as pd # Import pandas as it might be used here for DataFrame operations
from sklearn.model_selection import train_test_split

# The 'df' at this point contains all successfully downloaded and validated audio files.
# We will use this full dataset for splitting, disregarding class balance and minimum examples.

df = df.copy() # Explicitly work on a copy to avoid SettingWithCopyWarning

# Encode labels for all classes present in the full dataset
classes = sorted(df['common_name'].unique())
label2id = {c: i for i, c in enumerate(classes)}
id2label = {i: c for i, c in enumerate(classes)}
df['label'] = df['common_name'].map(label2id)

# Identify and remove classes with only one sample before splitting
# This is necessary for stratified splitting
class_counts = df['label'].value_counts()
single_sample_classes = class_counts[class_counts < 2].index

if not single_sample_classes.empty:
    # Explicitly make a copy to avoid SettingWithCopyWarning
    df = df[~df['label'].isin(single_sample_classes)].copy()
    print(f"Removed {len(single_sample_classes)} classes with less than 2 samples for stratified split.")
    # Re-encode labels if classes were removed to ensure contiguous labels
    classes = sorted(df['common_name'].unique())
    label2id = {c: i for i, c in enumerate(classes)}
    id2label = {i: c for i, c in enumerate(classes)}
    df['label'] = df['common_name'].map(label2id)

# Perform a simple train-validation split on the entire dataset
# Using a 80/20 split for training and validation
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Total samples in original DataFrame: {len(df)}")
print(f"Training samples (80%): {len(train_df)}")
print(f"Validation samples (20%): {len(val_df)}")
print(f"Number of classes: {len(classes)}")

In [ ]:
import torch
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader
import random
import logging

# Configure logging to catch errors without crashing the whole process
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# 4. Dataset and Augmentation
class FrogAudioDataset(Dataset):
    def __init__(self, df, is_train=True, augment_factor=1, target_sample_rate=16000, max_duration=5.0):
        self.df = df
        self.is_train = is_train
        self.augment_factor = augment_factor if is_train else 1
        self.target_sample_rate = target_sample_rate
        self.max_length = int(target_sample_rate * max_duration)

        self.mel_spectrogram = T.MelSpectrogram(
            sample_rate=target_sample_rate,
            n_fft=1024,
            hop_length=512,
            n_mels=64
        )
        self.amplitude_to_db = T.AmplitudeToDB()

        # SpecAugment techniques
        self.time_masking = T.TimeMasking(time_mask_param=30)
        self.freq_masking = T.FrequencyMasking(freq_mask_param=15)

    def __len__(self):
        return len(self.df) * self.augment_factor

    def __getitem__(self, idx):
        # Handle augmentation multiplier
        real_idx = idx % len(self.df)
        row = self.df.iloc[real_idx]

        try:
            waveform, sr = torchaudio.load(row['local_path'])

            # Resample if needed
            if sr != self.target_sample_rate:
                resampler = T.Resample(sr, self.target_sample_rate)
                waveform = resampler(waveform)

            # Convert stereo to mono
            if waveform.shape[0] > 1:
                waveform = torch.mean(waveform, dim=0, keepdim=True)

            # Pad or truncate to max_length
            if waveform.shape[1] > self.max_length:
                waveform = waveform[:, :self.max_length]
            elif waveform.shape[1] < self.max_length:
                pad_amount = self.max_length - waveform.shape[1]
                waveform = torch.nn.functional.pad(waveform, (0, pad_amount))

            # Generate Mel Spectrogram
            mel_spec = self.mel_spectrogram(waveform)
            mel_spec = self.amplitude_to_db(mel_spec)

            # Apply Augmentations (SpecAugment)
            if self.is_train and idx >= len(self.df): # Only augment the extra copies
                # Random noise
                if random.random() > 0.5:
                    noise = torch.randn_like(mel_spec) * 0.1
                    mel_spec = mel_spec + noise
                # Masking
                if random.random() > 0.5:
                    mel_spec = self.time_masking(mel_spec)
                if random.random() > 0.5:
                    mel_spec = self.freq_masking(mel_spec)

            # Multi-label one-hot encoding representation (to maximize identifying *any* frog)
            # Even if CSV is single label, we treat it as multi-label target
            label = torch.zeros(len(classes))
            label[row['label']] = 1.0

            return mel_spec, label

        except Exception as e:
            logging.error(f"Error loading or processing audio file {row['local_path']}: {e}")
            # Return None for problematic samples
            return None

# Custom collate_fn to filter out None values from the dataset
def collate_fn(batch):
    # Filter out None values
    batch = [item for item in batch if item is not None]
    if not batch:
        return None, None # Or handle empty batch as appropriate

    mel_specs, labels = zip(*batch)
    mel_specs = torch.stack(mel_specs)
    labels = torch.stack(labels)
    return mel_specs, labels

# 2x augmentation (1 original + 1 augmented copy per sample)
train_dataset = FrogAudioDataset(train_df, is_train=True, augment_factor=2)
val_dataset = FrogAudioDataset(val_df, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, collate_fn=collate_fn)

print(f"Total training examples after 2x augmentation: {len(train_dataset)}")

In [ ]:
import torch.nn as nn
import torchvision.models as models

# 5. Model Definition
class FrogClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # Use a lightweight ResNet
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        # Modify first layer to accept 1 channel (Mel Spectrogram)
        original_conv = self.backbone.conv1
        self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        # Copy weights from the first channel of the original model
        self.backbone.conv1.weight.data = original_conv.weight.data[:, :1, :, :]

        # Modify final fully connected layer
        num_ftrs = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(num_ftrs, num_classes)

    def forward(self, x):
        return self.backbone(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_classes = len(classes)
model = FrogClassifier(num_classes).to(device)


In [ ]:
from sklearn.metrics import f1_score, accuracy_score
import torch.optim as optim
import numpy as np

# 6. Training & Evaluation Loop
# Using BCEWithLogitsLoss because we want to maximize identifying any present species (multi-label objective)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3)
epochs = 15

for epoch in range(epochs):
    model.train()
    train_loss = 0
    for inputs, labels in train_loader:
        # Skip if batch is empty due to problematic files
        if inputs is None:
            continue

        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Evaluation
    model.eval()
    val_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in val_loader:
            # Skip if batch is empty due to problematic files
            if inputs is None:
                continue

            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            # Apply sigmoid and threshold at 0.5 to get predictions
            preds = (torch.sigmoid(outputs) > 0.5).float()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Check if all_preds or all_labels are empty before conversion
    if not all_preds or not all_labels:
        print(f"Epoch {epoch+1}/{epochs} - No valid samples for evaluation. Skipping metrics.")
        print(f"Train Loss: {train_loss/len(train_loader):.4f}")
        continue

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    # Calculate metrics
    # Handle cases where all_labels might be empty if all validation samples failed to load
    if len(all_labels) == 0:
        f1 = 0.0
        acc = 0.0
    else:
        f1 = f1_score(all_labels, all_preds, average='micro')
        acc = accuracy_score(all_labels, all_preds)

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(val_loader):.4f}")
    print(f"Val Micro F1: {f1:.4f} | Val Subset Accuracy: {acc:.4f}\n")

In [ ]:
# 7. Export to ONNX
onnx_file_path = os.path.join(BASE_DIR, "frog_classifier.onnx")

# Create a dummy input tensor matching the input shape (Batch Size, Channels, Mels, Time)
# E.g., 1 batch, 1 channel, 64 mels, 157 frames (depends on target_sample_rate and max_duration)
dummy_input = torch.randn(1, 1, 64, 157).to(device)

model.eval()
torch.onnx.export(
    model,
    dummy_input,
    onnx_file_path,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size', 3: 'time'},
                  'output': {0: 'batch_size'}}
)

print(f"Model successfully exported to {onnx_file_path}")
